<a href="https://colab.research.google.com/github/rpark3/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2013/Lab_13_The_Architecture_of_Dimensionality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/rpark3/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [5]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        19:48:47   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [8]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:56:58   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [12]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid
df['Price_Residuals']

,Price_Residuals
0,-56823.332740
1,19000.990249
2,-69149.815200
3,18481.157582
4,-2619.815668
...,...
995,21560.763160
996,-1940.516556
997,-3398.817768
998,2026.560249


In [13]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid
df['Age_Residuals']

,Age_Residuals
0,0.621803
1,-13.689856
2,3.233510
3,5.347789
4,4.053610
...,...
995,-14.814575
996,-6.511286
997,-1.986001
998,-4.589856


In [15]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [16]:
"""
3D Interactive OLS Regression Hyperplane Visualizer
====================================================
Multivariate OLS: Sale_Price ~ Property_Age + Distance_to_Tech_Hub
Uses statsmodels for regression and plotly.graph_objects for 3D visualization.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─────────────────────────────────────────────
# 1. SYNTHETIC DATA  (swap with your own df)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 200

Property_Age        = np.random.uniform(1, 40, n)       # years
Distance_to_Tech_Hub = np.random.uniform(0.5, 30, n)    # km

# True DGP: price falls with age & distance; add noise
Sale_Price = (
    800_000
    - 8_000  * Property_Age
    - 12_000 * Distance_to_Tech_Hub
    + np.random.normal(0, 40_000, n)
)

df = pd.DataFrame({
    "Sale_Price":          Sale_Price,
    "Property_Age":        Property_Age,
    "Distance_to_Tech_Hub": Distance_to_Tech_Hub,
})

# ─────────────────────────────────────────────
# 2. FIT THE OLS MODEL WITH STATSMODELS
# ─────────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
y = df["Sale_Price"]

model  = sm.OLS(y, X).fit()
print(model.summary())

# ── Extract coefficients from the fitted model ──────────────────────────────
# model.params is a pandas Series indexed by variable name.
# We pull each coefficient by name so the math below is fully explicit.
intercept   = model.params["const"]                  # β₀
coef_age    = model.params["Property_Age"]           # β₁
coef_dist   = model.params["Distance_to_Tech_Hub"]   # β₂
r_squared   = model.rsquared
adj_r2      = model.rsquared_adj

print(f"\nModel equation:")
print(f"  Sale_Price = {intercept:,.0f}"
      f" + ({coef_age:,.0f}) × Property_Age"
      f" + ({coef_dist:,.0f}) × Distance_to_Tech_Hub")

# ─────────────────────────────────────────────
# 3. BUILD THE REGRESSION SURFACE VIA MESHGRID
# ─────────────────────────────────────────────
# np.linspace creates evenly-spaced 1-D arrays spanning the observed range
# of each predictor.  Using the observed min/max keeps the surface anchored
# to the data — we don't extrapolate beyond what we've seen.
age_range  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(), 60)   # 60 pts along X-axis
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(), 60)  # 60 pts along Y-axis

# np.meshgrid converts two 1-D arrays into two 2-D arrays (AGE_GRID, DIST_GRID)
# so that every (row, col) pair encodes one (age, distance) combination.
# indexing='ij' → AGE_GRID[i,j] = age_range[i],  DIST_GRID[i,j] = dist_range[j]
AGE_GRID, DIST_GRID = np.meshgrid(age_range, dist_range, indexing="ij")

# Apply the OLS linear equation element-wise across the full grid.
# Each cell of PRICE_SURFACE holds the model's predicted Sale_Price
# for that (Property_Age, Distance_to_Tech_Hub) combination,
# holding all other variables constant (i.e., the ceteris-paribus plane).
PRICE_SURFACE = intercept + coef_age * AGE_GRID + coef_dist * DIST_GRID

# ─────────────────────────────────────────────
# 4. COMPUTE RESIDUALS FOR COLOUR-CODING POINTS
# ─────────────────────────────────────────────
df["Predicted"]  = model.fittedvalues
df["Residual"]   = model.resid          # positive = above plane, negative = below

# ─────────────────────────────────────────────
# 5. BUILD THE PLOTLY FIGURE
# ─────────────────────────────────────────────
fig = go.Figure()

# ── 5a. Regression hyperplane (Surface) ─────────────────────────────────────
fig.add_trace(go.Surface(
    x=AGE_GRID,
    y=DIST_GRID,
    z=PRICE_SURFACE,
    colorscale=[
        [0.0,  "rgba(99,  102, 241, 0.25)"],   # indigo, very transparent
        [0.5,  "rgba(139,  92, 246, 0.45)"],   # violet, semi-transparent
        [1.0,  "rgba(236,  72, 153, 0.65)"],   # pink,   more opaque
    ],
    showscale=False,
    opacity=0.72,
    name="OLS Regression Plane",
    hovertemplate=(
        "<b>Regression Plane</b><br>"
        "Property Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Predicted Price: $%{z:,.0f}<extra></extra>"
    ),
    contours=dict(
        z=dict(show=True, usecolormap=False,
               color="rgba(255,255,255,0.18)", width=1)
    ),
    lighting=dict(ambient=0.7, diffuse=0.5, roughness=0.6,
                  specular=0.3, fresnel=0.2),
))

# ── 5b. Actual data points, colour-coded by residual ────────────────────────
# Diverging colorscale: blue = over-predicted, red = under-predicted
max_resid = df["Residual"].abs().max()

fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    name="Observed Sale Prices",
    marker=dict(
        size=4.5,
        color=df["Residual"],
        colorscale="RdBu",
        cmin=-max_resid,
        cmax= max_resid,
        colorbar=dict(
            title=dict(text="Residual ($)", font=dict(size=12, color="#e2e8f0")),
            tickformat="$,.0f",
            thickness=14,
            len=0.55,
            x=1.02,
            tickfont=dict(color="#cbd5e1", size=10),
            outlinewidth=0,
        ),
        opacity=0.88,
        line=dict(width=0.4, color="rgba(255,255,255,0.5)"),
    ),
    hovertemplate=(
        "<b>Observed</b><br>"
        "Property Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Sale Price: $%{z:,.0f}<br>"
        "Residual: $%{marker.color:,.0f}<extra></extra>"
    ),
))

# ── 5c. Vertical residual lines (drop-lines to the plane) ───────────────────
# Each line runs from the actual point (z = Sale_Price) down/up to the plane
# (z = Predicted), making deviations visually obvious.
for _, row in df.iterrows():
    fig.add_trace(go.Scatter3d(
        x=[row["Property_Age"],   row["Property_Age"]],
        y=[row["Distance_to_Tech_Hub"], row["Distance_to_Tech_Hub"]],
        z=[row["Sale_Price"],     row["Predicted"]],
        mode="lines",
        line=dict(
            color="rgba(255,255,255,0.08)" if abs(row["Residual"]) < 20_000
            else "rgba(255,255,255,0.22)",
            width=1,
        ),
        showlegend=False,
        hoverinfo="skip",
    ))

# ─────────────────────────────────────────────
# 6. LAYOUT & STYLING
# ─────────────────────────────────────────────
equation_str = (
    f"Ŷ = {intercept:,.0f} "
    f"{'–' if coef_age < 0 else '+'} {abs(coef_age):,.0f}·Age "
    f"{'–' if coef_dist < 0 else '+'} {abs(coef_dist):,.0f}·Distance"
)

fig.update_layout(
    title=dict(
        text=(
            f"<b>Multivariate OLS Regression Hyperplane</b><br>"
            f"<span style='font-size:13px;color:#94a3b8'>"
            f"{equation_str} &nbsp;|&nbsp; "
            f"R² = {r_squared:.4f} &nbsp;|&nbsp; "
            f"Adj. R² = {adj_r2:.4f}</span>"
        ),
        x=0.5, xanchor="center",
        font=dict(size=18, color="#f1f5f9"),
    ),
    scene=dict(
        xaxis=dict(
            title="Property Age (years)",
            titlefont=dict(color="#94a3b8", size=12),
            tickfont=dict(color="#64748b", size=9),
            backgroundcolor="rgba(15,23,42,0.0)",
            gridcolor="rgba(148,163,184,0.12)",
            showbackground=True,
        ),
        yaxis=dict(
            title="Distance to Tech Hub (km)",
            titlefont=dict(color="#94a3b8", size=12),
            tickfont=dict(color="#64748b", size=9),
            backgroundcolor="rgba(15,23,42,0.0)",
            gridcolor="rgba(148,163,184,0.12)",
            showbackground=True,
        ),
        zaxis=dict(
            title="Sale Price ($)",
            titlefont=dict(color="#94a3b8", size=12),
            tickfont=dict(color="#64748b", size=9),
            tickformat="$,.0f",
            backgroundcolor="rgba(15,23,42,0.0)",
            gridcolor="rgba(148,163,184,0.12)",
            showbackground=True,
        ),
        bgcolor="rgba(2, 6, 23, 0.0)",
        camera=dict(
            eye=dict(x=1.55, y=-1.55, z=0.85),
            up=dict(x=0, y=0, z=1),
        ),
    ),
    paper_bgcolor="#020617",   # slate-950
    plot_bgcolor="#020617",
    legend=dict(
        x=0.01, y=0.97,
        font=dict(color="#cbd5e1", size=11),
        bgcolor="rgba(15,23,42,0.7)",
        bordercolor="rgba(148,163,184,0.2)",
        borderwidth=1,
    ),
    margin=dict(l=0, r=80, t=90, b=0),
    height=720,
)

# ─────────────────────────────────────────────
# 7. EXPORT
# ─────────────────────────────────────────────
output_path = "ols_3d_regression.html"
fig.write_html(
    output_path,
    include_plotlyjs="cdn",
    full_html=True,
    config={
        "displayModeBar": True,
        "modeBarButtonsToRemove": ["toImage"],
        "displaylogo": False,
        "scrollZoom": True,
    },
)
print(f"\n✓ Interactive plot saved → {output_path}")
fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.925
Model:                            OLS   Adj. R-squared:                  0.924
Method:                 Least Squares   F-statistic:                     1210.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          2.25e-111
Time:                        20:10:07   Log-Likelihood:                -2399.9
No. Observations:                 200   AIC:                             4806.
Df Residuals:                     197   BIC:                             4816.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 8.034e+05 